# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook guides you through loading and exploring the FAIR^2 dataset using the [`mlcroissant`](https://mlcommons.github.io/croissant) library, referencing dataset elements by their `@id` fields as per the Croissant schema.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant pandas matplotlib

## 1. Data Loading
Load the dataset metadata and explore the schema and top-level summary.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant JSON-LD schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Title: {metadata.name}\n")
print(f"Description: {metadata.description}\n")
print(f"Identifier: {metadata.identifier}")
print(f"Version: {metadata.version}")
print(f"License: {metadata.license}")

## 2. Data Overview
Review the available record sets, their details, and their unique `@id` values. All downstream exploration and extraction will reference the record sets by their `@id` fields, as required for working with Croissant datasets.

In [ ]:
# List all available record sets and their IDs with descriptions.
print("Available record sets (by @id):\n")
record_sets = list(metadata.record_sets)
for rs in record_sets:
    print(f"@id: {rs.id}")
    print(f"  name: {rs.name}")
    print(f"  description: {rs.description if hasattr(rs, 'description') else 'No description'}\n")

# For each record set, list its fields by @id, name, and data type.
for rs in record_sets:
    print(f"Fields for record set {rs.id}:")
    for field in rs.fields:
        print(f"  @id: {field.id}\n    name: {field.name}\n    dataType: {field.data_type}\n    description: {getattr(field, 'description', 'No description')}\n")

## 3. Data Extraction
Load the records from a chosen record set into a Pandas DataFrame for analysis, referencing all fields by their `@id`.

_In this example, we choose the main protein quantification record set. Please adjust the `main_record_set_id` variable below to your record set of interest._

In [ ]:
# Identify the @id for the main protein quantification table (likely the first record set, adjust as needed):
main_record_set = record_sets[0]  # Update this if other record set is primary
main_record_set_id = main_record_set.id

# Extract all data from the main record set
records = list(dataset.records(record_set=main_record_set_id))
df = pd.DataFrame(records)

print(f"Columns (@id) in DataFrame for {main_record_set_id}:")
print(list(df.columns))
df.head()

## 4. Exploratory Data Analysis (EDA)
Let's perform basic exploratory analysis:
- Filtering by a numeric field (e.g. MW (Molecular Weight), field referenced by its `@id`)
- Normalizing values
- Grouping by a categorical annotation (e.g. Sample Label, field referenced by `@id`)

_Replace `<numeric_field_id>` and `<group_field_id>` with a real field `@id` from the overview above!_


In [ ]:
# Example field IDs (update these according to the record set field listing!):
numeric_field_id = None
group_field_id = None

# Find likely numeric and grouping fields
for field in main_record_set.fields:
    if (field.data_type in ['Float', 'Integer']) and (not numeric_field_id):
        numeric_field_id = field.id
    if (field.data_type == 'Text' and 'label' in field.name.lower()) and (not group_field_id):
        group_field_id = field.id

print(f"Using numeric_field_id: {numeric_field_id}")
print(f"Using group_field_id: {group_field_id}")

# Proceed only if found
if numeric_field_id in df.columns:
    threshold = df[numeric_field_id].quantile(0.9)  # Top 10% as example
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    print(filtered_df[[numeric_field_id]].head())

    # Normalization (z-score)
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, norm_col]].head())

    # Groupby
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped (mean) {numeric_field_id} by {group_field_id}:")
        print(grouped_df.head())
else:
    print("No numeric field found for EDA. Please review field listing above and set `numeric_field_id`.")

## 5. Visualization
Visualize distributions and group differences using `matplotlib`. All plots should reference columns by their `@id`.

In [ ]:
import matplotlib.pyplot as plt

# Histogram for numeric field
if numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(7,4))
    df[numeric_field_id].hist(bins=40)
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.title(f"Distribution of {numeric_field_id}")
    plt.show()

    # If group_field_id is specified, boxplot by group
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10,5))
        df[[numeric_field_id, group_field_id]].boxplot(by=group_field_id, column=[numeric_field_id], grid=False)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.suptitle("")
        plt.ylabel(numeric_field_id)
        plt.xlabel(group_field_id)
        plt.show()

## 6. Conclusion

In this notebook, we've explored the FAIR<sup>2</sup> mass spectrometry dataset using the `mlcroissant` library, referencing record sets and fields by their Croissant `@id`. We loaded and reviewed the dataset's metadata, explored its schema, loaded records, and performed basic data filtering, normalization, and visualization. For more advanced analysis, refer to the full Croissant schema and documentation for further fields, relations, and usage notes.